In [ ]:
folder = ''

In [ ]:
# load
import json
import pathlib
import pandas as pd

folder = pathlib.Path(folder)
assert folder.exists()

# aggregate results into csv necessary
file_list = list(folder.glob('out*.json'))
dict_list = list()
for file in file_list:
    with open(file, 'r') as f:
        dict_list.append(json.load(f))
    
# load aggregated results
f_csv = folder / 'results.csv'
if f_csv.exists():
    df = pd.read_csv(f_csv, index_col=None)
else:
    df = pd.DataFrame()

# add in existing result
df = pd.concat((df, pd.DataFrame(dict_list)))

# round p_value to 14 decimal places (avoids floating point comparison failure)
df['p_val'] = df['p_val'].round(14)

# drop duplicates & check for conflicting results
df.drop_duplicates(inplace=True)
assert df.value_counts(subset=['p_val', 'seed', 'Analysis']).max() == 1
    
# overwrite csv with latest / greatest
df.to_csv(f_csv, index=False)

# delete json files (they're in csv)
for file in file_list:
    file.unlink()

In [ ]:
import numpy as np
from collections import defaultdict


# extract
pval_list = sorted(df['p_val'].unique())
seed_list = sorted(df['seed'].unique())

shape = len(seed_list), len(pval_list)
score_dict = defaultdict(lambda: np.full(shape=shape, fill_value=np.nan))

for _, row in df.iterrows():
    seed_idx = seed_list.index(row['seed'])
    pval_idx = pval_list.index(row['p_val'])
    
    for feat in ('f1', 'sens', 'spec'):
        score_dict[row['Analysis'], feat][seed_idx, pval_idx] = row[feat]

In [ ]:
# plot
import seaborn as sns
import matplotlib.pyplot as plt

sns.set()

fig, ax = plt.subplots(2, 3)

# plot top row
style_dict = {'AnalysisTFCE': {'color': 'r'}, 
              'AnalysisHGLM': {'color': 'b'}}
style_single = {'linewidth': .5,
                'zorder': 1,
                'label': '_nolegend_'}
style_mean = {'linewidth': 5,
              'zorder': 2,
              'label': '_nolegend_'}
for _ax, feat in zip(ax[0, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    for method, kwargs in style_dict.items():
        plt.plot(pval_list, score_dict[method, feat].T, **kwargs, **style_single)
        plt.plot(pval_list, np.nanmean(score_dict[method, feat], axis=0), **kwargs, **style_mean)

    plt.xlabel('p_val')
    plt.ylabel(feat)
    plt.xscale('log')

# plot bottom row
for _ax, feat in zip(ax[1, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    x = score_dict['AnalysisHGLM', feat] - score_dict['AnalysisTFCE', feat]
    plt.axhline([0], linewidth=2, color='k')
    plt.plot(pval_list, x.T, color='k', **style_single)
    plt.plot(pval_list, np.nanmean(x, axis=0), color='k', **style_mean)

    plt.xlabel('p_val')
    plt.ylabel(f'{feat}: hglm - TFCE')
    plt.xscale('log')
    
# add legend in last plot of top row
plt.sca(ax[0, -1])
del style_single['label']
for method, kwargs in style_dict.items():
    plt.plot([], [], label=method[-4:], **kwargs, **style_single)
plt.legend()
    
fig.set_size_inches(10, 6)
fig.tight_layout()
fig.savefig(folder / 'hglm_vs_TFCE.png', bbox_inches='tight')

# Compare Timing

Seconds per analysis run

In [ ]:
sns.swarmplot(data=df, x='time_sec', hue='Analysis');

# Examine case where TFCE has greatest advantage of HRBA



In [ ]:
import cloudpickle as pickle
import gzip

def get_uuid(**match_dict):
    """ gets series of all uuid values matching the input dict """
    s_bool = pd.Series(True, index=df.index)
    for col, val in match_dict.items():
        s_bool &= df[col] == val
    return df[s_bool]['uuid']

def load(uuid=None, **kwargs):
    """ loads detail file """
    if uuid is None:
        s_uuid = get_uuid(**kwargs)
        assert s_uuid.size == 1, f'search found {s_uuid.size} unique experiments'
        uuid = s_uuid.iloc[0]
        
    file_list = list(folder.glob(f'*{uuid}_detail*'))
    assert len(file_list) == 1
    file = file_list[0]
    
    with gzip.open(file, 'rb') as f:
        x = pickle.load(f)

    return x

# sort by f1 difference pval_idx[0] and seed_idx[0] give the pvalue and seed yielding 
# largest difference (TFCE outperforming HGLM)
f1_diff = score_dict['AnalysisHGLM', 'f1'] - score_dict['AnalysisTFCE', 'f1']
seed_idx, pval_idx = np.unravel_index(np.argsort(f1_diff.flatten()), f1_diff.shape)
pval_seed_sorted = [(pval_list[pval_idx[idx]], seed_list[seed_idx[idx]]) for idx in range(seed_idx.size)]

In [ ]:
from hglm.plot import scatter_size_vs_stat
from hglm.graph import get_f1
from IPython.display import display

p_val, seed = pval_seed_sorted[0]

ana_hglm, effect = load(p_val=p_val, seed=seed, Analysis='AnalysisHGLM')

display(df[(df['p_val'] == p_val) & (df['seed'] == seed)])

fig, ax = plt.subplots(3, 1)
plt.sca(ax[0])
scatter_size_vs_stat(analysis=ana_hglm, y_feat=ana_hglm.llr, mask=effect.mask)
line_size = np.geomspace(1, (ana_hglm.exp.mask_idx >= 0).sum(), 100)
line_llr = ana_hglm.llr_predict(ana_hglm.llr_beta, line_size)
plt.plot(line_size, line_llr)
plt.ylabel('llr')

plt.sca(ax[1])
scatter_size_vs_stat(analysis=ana_hglm, y_feat=ana_hglm.llr_adjust, mask=effect.mask)
plt.ylabel('llr-adjusted')

plt.sca(ax[2])
f1 = get_f1(mask=effect.mask,
            mask_idx=ana_hglm.exp.mask_idx,
            children=ana_hglm.child_dict[0])
plt.scatter(ana_hglm.size[0, :], ana_hglm.p_val, c=f1, cmap='plasma')
plt.xscale('log')
plt.xlabel('size')
plt.ylabel('p-value')
fig.set_size_inches(5, 10)
fig.tight_layout()

import hglm

miss, hits = hglm.graph.get_miss_hits(children=ana_hglm.child_dict[0], mask_idx=ana_hglm.exp.mask_idx, mask=effect.mask)
f1 = hglm.graph.get_f1(children=ana_hglm.child_dict[0], mask_idx=ana_hglm.exp.mask_idx, mask=effect.mask)

# reg_idx, miss, hits, pval, pval_fwer
for idx, effect_est in enumerate(ana_hglm.effect_list):
    print(f'discovered effect {idx}:')
    reg_idx = effect_est.reg_idx
    print(f'region {reg_idx} dice: {f1[reg_idx]:.3f} true pos voxels: {hits[reg_idx]:.0f} false pos voxels: {miss[reg_idx]:.0f} pval: {effect_est.p_val:.3e} pval_fwer: {effect_est.p_val_fwer:.3e}')
    